GridSearch

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# Total combinations = 3 * 3 * 2 = 18 models
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,                 # 5-fold cross-validation (90 total fits)
    scoring='accuracy'
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Output optimal parameters
print("Best Hyperparameters:", grid_search.best_params_)
print("Best Cross-Validation Score:", grid_search.best_score_)

Transfer Learning Feature Extraction

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 1. Load pre-trained MobileNetV2 without its original classification head (include_top=False)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Freeze the entire base model architecture
base_model.trainable = False

# 3. Construct the network with a custom classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),  # Flattens spatial map into 1D feature vector
    layers.Dropout(0.3),              # Regularization layer
    layers.Dense(2, activation='softmax')  # Binary/Custom output (e.g., Cat vs Dog)
])

# 4. Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 5. Train ONLY the top classification head
# history = model.fit(train_ds, validation_data=val_ds, epochs=10)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Transfer Learning - Fine Tuning

In [3]:
# 1. Unfreeze the entire base model to configure specific layer access
base_model.trainable = True

# 2. Freeze all layers EXCEPT the last 20 layers
fine_tune_at = len(base_model.layers) - 20

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# 3. Re-compile the model with a MUCH SMALLER learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Reduced lr for safety
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Continue training (Fine-Tuning stage)
# fine_tune_history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10,
#     initial_epoch=history.epoch[-1]
# )

CNN - LeNet Architecture

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_lenet5(input_shape=(32, 32, 1), num_classes=10):
    model = models.Sequential([
        # C1: Conv Layer
        layers.Conv2D(6, kernel_size=(5, 5), strides=(1, 1), activation='relu', input_shape=input_shape),
        # S2: Average Pooling
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),

        # C3: Conv Layer
        layers.Conv2D(16, kernel_size=(5, 5), strides=(1, 1), activation='relu'),
        # S4: Average Pooling
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),

        # Flatten for Dense Layers (replaces C5/Dense transition)
        layers.Flatten(),

        # F6: Fully Connected
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),

        # Output Layer
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_lenet5()
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

CNN - AlexNet Architecture

In [5]:

import tensorflow as tf
from tensorflow.keras import layers, models

def build_alexnet(input_shape=(227, 227, 3), num_classes=1000):
    model = models.Sequential([
        # Layer 1: Conv1 + MaxPool
        layers.Conv2D(96, kernel_size=(11, 11), strides=4, activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(3, 3), strides=2),

        # Layer 2: Conv2 + MaxPool
        layers.Conv2D(256, kernel_size=(5, 5), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(3, 3), strides=2),

        # Layer 3, 4, 5: Conv3, Conv4, Conv5 + MaxPool
        layers.Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.Conv2D(384, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.Conv2D(256, kernel_size=(3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(3, 3), strides=2),

        # Flatten and Dense Classifier
        layers.Flatten(),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_alexnet()
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1000)           │     4,097,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,378,344 (237.95 MB)

 Trainable params: 62,378,344 (237.95 MB)

 Non-trainable params: 0 (0.00 B)

RNN

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Simple Many-to-One Sentiment Classifier using SimpleRNN / LSTM
model = models.Sequential([
    # Input: Sequence of 100 word indices
    layers.Embedding(input_dim=10000, output_dim=64, input_length=100),

    # Recurrent Layer (e.g., SimpleRNN, LSTM, or GRU)
    layers.SimpleRNN(64, return_sequences=False),

    # Dense Classifier
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Prepare Synthetic Data (5 sequences of word IDs)
# Numbers represent specific words in a vocabulary (e.g., 2="great", 3="bad", etc.)
X_raw = [
    [2, 4, 5, 8],       # "great movie really loved" -> Positive (1)
    [3, 6, 7, 9, 10],   # "bad plot terrible acting horrible" -> Negative (0)
    [2, 2, 4],          # "great great movie" -> Positive (1)
    [3, 7, 9],          # "bad terrible acting" -> Negative (0)
    [2, 5, 4, 8]        # "great really movie loved" -> Positive (1)
]
y_train = np.array([1, 0, 1, 0, 1])  # Target labels (1 = Positive, 0 = Negative)

# 2. Pad sequences so all inputs have the same length (max_len = 6)
max_len = 6
X_train = pad_sequences(X_raw, maxlen=max_len, padding='pre')

# 3. Build the RNN Architecture
vocab_size = 50   # Size of dictionary/vocabulary
embedding_dim = 16 # Dimensionality of word embeddings
rnn_units = 32     # Number of hidden units in the SimpleRNN layer

model = models.Sequential([
    # Embedding Layer: Maps word indices to dense vectors of shape (batch, 6, 16)
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),

    # SimpleRNN Layer: Processes sequence step-by-step, returning final hidden state vector
    layers.SimpleRNN(units=rnn_units, return_sequences=False, activation='tanh'),

    # Output Layer: Binary classification (0 to 1)
    layers.Dense(1, activation='sigmoid')
])

# 4. Compile the Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 5. Train the Model
model.fit(X_train, y_train, epochs=20, verbose=0)

# 6. Test Prediction on New Data
test_review = [[2, 4, 8]]  # "great movie loved"
test_padded = pad_sequences(test_review, maxlen=max_len, padding='pre')
prediction = model.predict(test_padded, verbose=0)

print(f"Predicted Probability: {prediction[0][0]:.4f}")
print("Sentiment Class:", "Positive" if prediction[0][0] > 0.5 else "Negative")

Predicted Probability: 0.6385
Sentiment Class: Positive
